In [1]:
import pandas as pd
import numpy as np
import time
import os
import pickle
from geopy.geocoders import Nominatim
from tqdm import tqdm
import warnings
import re
warnings.filterwarnings('ignore')


In [2]:
df_initial = pd.read_csv('data/schools_filtered_2.csv')

In [3]:
df_initial

,login,locality_type,locality_name,population,correction,pupils_amount,district
0,sch05153601,НПСТ,Червленные-буруны,1.000-10.000,1,287,СКФО
1,sch05153598,НПСТ,Ортатюбе,1.000-10.000,0,110,СКФО
2,sch05153594,НПСТ,Кумли,0-1.000,0,96,СКФО
3,sch05153592,НПСТ,Карагас,1.000-10.000,0,220,СКФО
4,sch05153591,НПСТ,Калининаул,0-1.000,0,92,СКФО
...,...,...,...,...,...,...,...
13622,sch16163045,НПСТ,Большой Сухояш,0-1.000,0,38,ПФО
13623,sch27173299,НПСТ,село Малышево,0-1.000,1,110,ДВФО
13624,sch30103082,НПСТ,Заречное,1.000-10.000,0,225,ЮФО
13625,sch05153226,НПСТ,c. Кунки,0-1.000,0,21,СКФО


In [4]:
df_addr = pd.read_excel('data/addresses.xlsx', header=0)            # таблица с адресами


In [5]:
df_addr

,Логин ОО,Адрес
0,sch01110001,385601 РА Гиагинский район ст.Гиагинская ул.Ле...
1,sch01110003,385600 РА Гиагинский район ст.Гиагинская ул.Бо...
2,sch01110004,385600 РА Гиагинский район ст.Гиагинская ул.Кр...
3,sch01110005,385631 РА Гиагинский район п.Гончарка ул.Гиаги...
4,sch01110006,"385634, РА Гиагинский район х.Прогресс ул. Цен..."
...,...,...
39649,sch92126013,"299058, г. Севастополь, пр.Античный, строение"
39650,sch92126015,"299021, г.Севастополь, ул.Тараса Шевченко,"
39651,sch92126016,"299053, Российская Федерация, город Севастопол..."
39652,sch92120001,"299011, г. Севастополь, ул. Попова,"


In [ ]:
valid_logins = set(df_initial['login'])
df_filtered = df_addr[df_addr['Логин ОО'].isin(valid_logins)].copy()
print(f" Найдено {len(df_filtered)} адресов для школ из исходного датасета.")


In [7]:
def clean_address(addr):
    if pd.isna(addr): return ""
    addr = str(addr).strip()
    # Убираем 6-значный почтовый индекс в начале (OSM он не нужен, иногда мешает)
    if len(addr) > 6 and addr[:6].isdigit() and addr[6] == ' ':
        addr = addr[7:]
    # Нормализуем пробелы и регистр
    addr = ' '.join(addr.split())
    return addr

In [8]:
df_filtered['addr_clean'] = df_filtered['Адрес'].apply(clean_address)


In [9]:
df_filtered

,Логин ОО,Адрес,addr_clean
0,sch01110001,385601 РА Гиагинский район ст.Гиагинская ул.Ле...,РА Гиагинский район ст.Гиагинская ул.Ленина
1,sch01110003,385600 РА Гиагинский район ст.Гиагинская ул.Бо...,РА Гиагинский район ст.Гиагинская ул.Боевая
2,sch01110004,385600 РА Гиагинский район ст.Гиагинская ул.Кр...,РА Гиагинский район ст.Гиагинская ул.Красная
3,sch01110005,385631 РА Гиагинский район п.Гончарка ул.Гиаги...,РА Гиагинский район п.Гончарка ул.Гиагинская
4,sch01110006,"385634, РА Гиагинский район х.Прогресс ул. Цен...","385634, РА Гиагинский район х.Прогресс ул. Цен..."
...,...,...,...
16912,sch35153158,"г. Череповец, ул. Металлургов,","г. Череповец, ул. Металлургов,"
16913,sch35153159,"г. Череповец, ул. Набережная,","г. Череповец, ул. Набережная,"
16914,sch35153160,"г. Череповец, ул. Вологодская,","г. Череповец, ул. Вологодская,"
16915,sch35153161,"г. Череповец, ул. Моченкова,","г. Череповец, ул. Моченкова,"


In [10]:
import re
import pandas as pd

def clean_address_to_city_street(address: str) -> str:
    """
    Преобразует сырой адрес в формат "Населённый пункт, Улица".
    Не повреждает названия городов, содержащие "ул", "респ" и т.п.
    """
    if not isinstance(address, str) or not address.strip():
        return ""

    s = address.strip()
    # 1. Убираем почтовый индекс в начале строки (6 цифр + возможные запятые/пробелы)
    s = re.sub(r'^\d{6}[,\s]*', '', s)

    # Паттерны для населённых пунктов (с префиксом)
    sett_prefix = r'(?:г\.?\s*|город\s+|с\.?\s*|село\s+|п\.?\s*|поселок\s+|р\.п\.?\s*|ст\.?\s*|станица\s+|х\.?\s*|хутор\s+|д\.?\s*|деревня\s+|аул\s+|мкр\.?\s*|микрорайон\s+|квартал\s+|поселение\s+)'

    # Паттерны для улиц (с префиксом)
    street_prefix = r'(?:ул\.?\s*|улица\s+|пер\.?\s*|переулок\s+|пр-т\.?\s*|проспект\s+|ш\.?\s*|шоссе\s+|б-р\.?\s*|бульвар\s+|наб\.?\s*|набережная\s+|аллея\s+|проезд\s+|тупик\s+|пл\.?\s*|площадь\s+|тракт\s+|линия\s+)'

    # 2. Ищем улицу. Поддерживаем два формата: "ул. Название" и "Название улица"
    street_match_pre = re.search(rf'{street_prefix}([А-Яа-яёЁА-Я\s\-\.]+?)(?:\s*,|\s+дом|\s+д\.|\s+корп|\s+строение|\s+помещение|\s+пом|лит|$)', s, re.IGNORECASE)
    if street_match_pre:
        street = street_match_pre.group(1).strip().rstrip(',.')
    else:
        street_match_suf = re.search(r'([А-Яа-яёЁА-Я\s\-]+?)\s*(?:улица|переулок|проспект|шоссе|бульвар|набережная|аллея|проезд|тупик|площадь|тракт|линия)(?:\s*,|\s+дом|\s+д\.|\s+корп|\s+строение|\s+помещение|\s+пом|лит|$)', s, re.IGNORECASE)
        street = street_match_suf.group(1).strip().rstrip(',.') if street_match_suf else ""

    # Определяем область поиска города (всё, что стоит ДО улицы)
    search_area = s
    if street_match_pre:
        search_area = s[:street_match_pre.start()]
    elif street_match_suf:
        search_area = s[:street_match_suf.start()]

    # 3. Ищем населённый пункт
    # Сначала ищем с явными префиксами
    sett_matches = list(re.finditer(rf'{sett_prefix}([А-Яа-яёЁА-Я\s\-]+?)(?:\s*,|\s*{street_prefix}|$)', search_area, re.IGNORECASE))
    if sett_matches:
        # Берём последний найденный НП (обычно он ближе к улице и точнее)
        settlement = sett_matches[-1].group(1).strip().rstrip(',.')
    else:
        # Фоллбэк: если префикса нет, берём первое слово с заглавной буквы до запятой или улицы
        fallback = re.search(r'([А-Яа-яёЁА-Я][А-Яа-яёЁА-Я\s\-]*?)(?:\s*,|\s*{street_prefix}|$)', search_area, re.IGNORECASE)
        settlement = fallback.group(1).strip().rstrip(',.') if fallback else ""

    # Нормализуем внутренние пробелы
    settlement = re.sub(r'\s+', ' ', settlement)
    street = re.sub(r'\s+', ' ', street)

    # 4. Формируем результат
    if settlement and street:
        return f"Россия, {settlement}, {street}"
    elif settlement:
        return None #f"Россия, {settlement}"
    elif street:
        return None #f"Россия, {street}"

    return None #f"Россия, {s}"  # Возвращаем исходное, если ничего не удалось распознать

In [ ]:
df_filtered["addr_clean"] = df_filtered["Адрес"].apply(clean_address_to_city_street)

# Смотрим результат и долю пропусков
print(df_filtered[["Адрес", "addr_clean"]].head(10))
print(f"\nУспешно обработано: {df_filtered['addr_clean'].notna().sum()} из {len(df_filtered)} ({df_filtered['addr_clean'].notna().mean()*100:.1f}%)")

In [12]:
df_filtered = df_filtered.dropna()

In [ ]:
CACHE_FILE = "geocode_cache_addresses.pkl"
cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, "rb") as f:
            cache = pickle.load(f)
    except Exception:
        print("Кэш повреждён, создаётся новый.")

geolocator = Nominatim(user_agent="diploma_schools_v4")
tqdm.pandas(desc="Геокодирование адресов")

def geocode_with_metadata(addr):
    if addr in cache:
        return cache[addr]

    try:
        #loc = geolocator.geocode("Москва, Россия")
        loc = geolocator.geocode(addr, exactly_one=True, timeout=10)
        if loc:
            raw = loc.raw
            address_comp = raw.get('address', {})
            res = {
                'lat': loc.latitude,
                'lon': loc.longitude,
                #'display_name': loc.display_name,
                'osm_id': raw.get('osm_id'),
                'osm_type': raw.get('osm_type'),  # node/way/relation
                'place_rank': raw.get('place_rank', 0),
                'importance': raw.get('importance', 0.0),
                'postcode': address_comp.get('postcode', ''),
                'city': address_comp.get('city') or address_comp.get('town') or address_comp.get('village') or '',
                'road': address_comp.get('road', ''),
                'house_number': address_comp.get('housenumber', ''),
                'success': True
            }
        else:
            res = {k: np.nan for k in ['lat','lon','display_name','osm_id','osm_type','place_rank','importance','postcode','city','road','house_number']}
            res['success'] = False
    except Exception as e:
        res = {k: np.nan for k in ['lat','lon','display_name','osm_id','osm_type','place_rank','importance','postcode','city','road','house_number']}
        res['success'] = str(e)

    cache[addr] = res
    time.sleep(1.1)  # лимит Nominatim: 1 запрос/сек
    return res

In [48]:
df_filtered[df_filtered['Логин ОО'].isna()]

,Логин ОО,Адрес,addr_clean


In [ ]:
results = df_filtered['addr_clean'].progress_apply(geocode_with_metadata)

Геокодирование адресов: 100%|██████████| 11819/11819 [3:29:17<00:00,  1.06s/it] 


In [54]:
df_geo_final = df_filtered

In [59]:
df_geo_final['lat'] = [elem['lat'] for elem in results.to_list()]
df_geo_final['lon'] = [elem['lon'] for elem in results.to_list()]

In [62]:
df_geo_final.dropna().to_csv('schools_filtered_with_loc.csv', index=False)